In [ ]:
!pip install -q fastapi uvicorn python-multipart pymupdf python-docx ollama requests nest-asyncio pyngrok

In [ ]:
import subprocess
import time
import requests

# Iniciar servidor Ollama
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Verificar servidor
resposta = requests.get("http://127.0.0.1:11434")

print("Ollama iniciado:", resposta.status_code == 200)

In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
import ollama

MODELO = "qwen2.5:3b"


def consultar_ia(prompt):

    resposta = ollama.chat(
        model=MODELO,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return resposta["message"]["content"]


def simplificar_texto(texto):

    prompt = f"""
    Você é um assistente pedagógico especializado
    em acessibilidade educacional.

    Simplifique o texto para uma aluna do 8º ano.

    Utilize:
    - Linguagem simples;
    - Frases curtas;
    - Ordem direta;
    - Vocabulário acessível;
    - Preservação das informações importantes.

    Texto original:
    {texto}

    Texto simplificado:
    """

    return consultar_ia(prompt)


def organizar_conteudos(texto):

    prompt = f"""
    Analise o material didático abaixo.

    Identifique:
    1. Disciplina;
    2. Conteúdos principais;
    3. Subtópicos;
    4. Objetivos de aprendizagem.

    Organize a resposta de maneira clara e estruturada.

    Material:
    {texto}
    """

    return consultar_ia(prompt)


def gerar_planejamento(conteudos, semestre):

    prompt = f"""
    Você é um assistente pedagógico.

    Crie um planejamento didático para o semestre:
    {semestre}

    Distribua os conteúdos em uma sequência lógica.

    Para cada período, apresente:
    - Conteúdo;
    - Objetivo de aprendizagem;
    - Sugestão de atividade.

    Utilize uma linguagem clara e organizada.

    Conteúdos:
    {conteudos}
    """

    return consultar_ia(prompt)


print("Funções de inteligência artificial carregadas!")

In [ ]:
import io
import json
import os
import fitz
import ollama

from datetime import datetime

from docx import Document
from fastapi import FastAPI, File, UploadFile, HTTPException
from pydantic import BaseModel


app = FastAPI(
    title="EduAcessível API",
    description="API de organização e adaptação de materiais educacionais",
    version="1.0.0"
)


# HISTÓRICO
ARQUIVO_HISTORICO = "historico.json"


def carregar_historico():

    if os.path.exists(ARQUIVO_HISTORICO):

        with open(ARQUIVO_HISTORICO, "r", encoding="utf-8") as arquivo:
            return json.load(arquivo)

    return []


def salvar_historico(dados):

    with open(ARQUIVO_HISTORICO, "w", encoding="utf-8") as arquivo:
        json.dump(dados, arquivo, ensure_ascii=False, indent=4)


historico = carregar_historico()


# EXTRAÇÃO DE DOCUMENTOS

def extract_text_from_pdf(file_bytes):

    doc = fitz.open(stream=file_bytes, filetype="pdf")

    return "\n".join(
        page.get_text() for page in doc
    ).strip()


def extract_text_from_docx(file_bytes):

    doc = Document(io.BytesIO(file_bytes))

    return "\n".join(
        p.text for p in doc.paragraphs if p.text.strip()
    )


# MODELO DE IA

MODELO = "qwen2.5:3b"


def consultar_ia(prompt):

    resposta = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return resposta["message"]["content"]


# 1. UPLOAD

@app.post("/file/upload")
async def upload_file(file: UploadFile = File(...)):

    filename = file.filename or ""
    ext = filename.rsplit(".", 1)[-1].lower()

    if ext not in ["pdf", "docx", "txt"]:
        raise HTTPException(
            status_code=400,
            detail="Formato não suportado."
        )

    file_bytes = await file.read()

    if len(file_bytes) > 10 * 1024 * 1024:
        raise HTTPException(
            status_code=413,
            detail="Arquivo maior que 10 MB."
        )

    try:

        if ext == "pdf":
            texto = extract_text_from_pdf(file_bytes)

        elif ext == "docx":
            texto = extract_text_from_docx(file_bytes)

        else:
            texto = file_bytes.decode("utf-8-sig")

    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Não foi possível ler o documento."
        )

    if not texto.strip():
        raise HTTPException(
            status_code=400,
            detail="Nenhum texto foi encontrado no documento."
        )

    return {
        "arquivo": filename,
        "quantidade_caracteres": len(texto),
        "texto": texto
    }


# 2. SIMPLIFICAÇÃO

class TextoEntrada(BaseModel):
    texto: str


@app.post("/ia/simplificar")
def simplificar(dados: TextoEntrada):

    if not dados.texto.strip():
        raise HTTPException(400, "Informe um texto.")

    resultado = consultar_ia(f"""
    Simplifique o texto abaixo para uma aluna do 8º ano.
    Utilize frases curtas, linguagem acessível e preserve
    as informações importantes.

    Texto:
    {dados.texto}
    """)

    return {
        "texto_original": dados.texto,
        "texto_simplificado": resultado
    }


# 3. ORGANIZAÇÃO DOS CONTEÚDOS

@app.post("/ia/organizar")
def organizar(dados: TextoEntrada):

    resultado = consultar_ia(f"""
    Analise o material didático e identifique:
    - Disciplina;
    - Conteúdos principais;
    - Subtópicos;
    - Objetivos de aprendizagem.

    Material:
    {dados.texto}
    """)

    return {
        "conteudos_organizados": resultado
    }


# 4. PLANEJAMENTO SEMESTRAL

class PlanejamentoEntrada(BaseModel):
    conteudos: str
    semestre: str = "Primeiro semestre"


@app.post("/ia/planejamento")
def planejamento(dados: PlanejamentoEntrada):

    resultado = consultar_ia(f"""
    Crie um planejamento didático para {dados.semestre}.

    Distribua os conteúdos em uma sequência lógica.
    Para cada período, apresente:
    - Conteúdo;
    - Objetivo de aprendizagem;
    - Sugestão de atividade.

    Conteúdos:
    {dados.conteudos}
    """)

    registro = {
        "id": len(historico) + 1,
        "data": datetime.now().isoformat(),
        "semestre": dados.semestre,
        "planejamento": resultado
    }

    historico.append(registro)
    salvar_historico(historico)

    return registro


# 5. HISTÓRICO

@app.get("/historico")
def consultar_historico():

    return {
        "total": len(historico),
        "historico": historico
    }


# VERIFICAÇÃO DA API

@app.get("/")
def inicio():

    return {
        "mensagem": "EduAcessível API funcionando!",
        "versao": "1.0.0"
    }

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

# Informe seu token do Ngrok
ngrok.set_auth_token("SEU_TOKEN_NGROK")

# Criar endereço público
public_url = ngrok.connect(8000)

print("API disponível em:", public_url)

# Iniciar FastAPI
def iniciar_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = threading.Thread(
    target=iniciar_api,
    daemon=True
)

thread.start()